# Compare quoter output to VP base price

Generate quoter config, then run ChatbotProjectQuoter one window at a time and compare unit_price to total_base_price.

In [10]:
# Cell 1: Generate the config object
import importlib
import json
import sys
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "qa" else Path.cwd()
sys.path.insert(0, str(root))

import qa.vp_quotes_to_quoter_config as _qa_config
importlib.reload(_qa_config)
from qa.vp_quotes_to_quoter_config import convert_all

input_path = root / "validation_quotes" / "2026" / "parsed_all_vp_quotes_2026.json"
pricing_path = root / "valid_config_generator" / "pricing.yaml"
config = convert_all(str(input_path), str(pricing_path))

# Save for inspection
out_path = root / "validation_quotes" / "2026" / "parsed_all_vp_quotes_2026_quoter_config.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
print(f"Saved config to {out_path}")

# Windows only (skip installation_required)
windows = {k: v for k, v in config.items() if k.startswith("window_")}
print(f"Config has {len(windows)} windows")

Saved config to /Users/km/ai-estimator/validation_quotes/2026/parsed_all_vp_quotes_2026_quoter_config.json
Config has 98 windows


In [11]:
# Cell 2: One window at a time -> project quoter -> compare unit_price to base_price_total
import sys
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "qa" else Path.cwd()
sys.path.insert(0, str(root))

from chatbot_project_quoter.chatbot_project_quoter import ChatbotProjectQuoter

quoter = ChatbotProjectQuoter()
results = []

for win_key, win_entry in windows.items():
    single_config = {
        win_key: {
            "config": win_entry["config"],
            "quantity": win_entry["quantity"],
        },
        "installation_required": False,
    }
    base_price_total = win_entry.get("base_price_total")  # VP quote line total
    try:
        total, display_dict, _ = quoter.quote_project(single_config, format="string")
        bd = display_dict.get("breakdown", {}).get(win_key)
        if bd is None:
            results.append({"window": win_key, "error": "no breakdown", "base_price_total": base_price_total})
            continue
        unit_price = bd.get("unit_price")  # quoter cost per unit (after discount)
        qty = bd.get("quantity", 1)
        quoter_total = (unit_price or 0) * qty  # quoter window total (after discount)
        diff = (quoter_total - base_price_total) if base_price_total is not None else None
        results.append({
            "window": win_key,
            "line_no": win_entry.get("line_no"),
            "source_file": win_entry.get("source_file"),
            "base_price_total": base_price_total,
            "quoter_unit_price": unit_price,
            "quoter_total": quoter_total,
            "quantity": qty,
            "diff": diff,
        })
    except Exception as e:
        results.append({"window": win_key, "error": str(e), "base_price_total": base_price_total})

# Summary: compare quoter_total vs base_price_total
import pandas as pd
df = pd.DataFrame([r for r in results if "error" not in r])
if len(df) and "diff" in df.columns:
    df["diff_pct"] = (df["diff"] / df["base_price_total"] * 100).round(2)
display(df)
errors = [r for r in results if "error" in r]
if errors:
    print(f"Errors ({len(errors)}):", errors[:5])

,window,line_no,source_file,base_price_total,quoter_unit_price,quoter_total,quantity,diff,diff_pct
0,window_1,1,PO#_14FIELDING_Order_519307.PDF,641.47,681.150750,681.150750,1,39.680750,6.19
1,window_2,3,PO#_14FIELDING_Order_519307.PDF,625.74,681.150750,681.150750,1,55.410750,8.86
2,window_3,1,PO#_14FIELDING_Order_519310.PDF,448.06,474.245625,474.245625,1,26.185625,5.84
3,window_4,2,PO#_14FIELDING_Order_519310.PDF,448.06,474.245625,474.245625,1,26.185625,5.84
4,window_5,3,PO#_14FIELDING_Order_519310.PDF,448.06,474.245625,474.245625,1,26.185625,5.84
...,...,...,...,...,...,...,...,...,...
93,window_94,12,PO#_STK101225_Order_518692.PDF,200.72,197.780450,197.780450,1,-2.939550,-1.46
94,window_95,13,PO#_STK101225_Order_518692.PDF,200.72,197.780450,197.780450,1,-2.939550,-1.46
95,window_96,14,PO#_STK101225_Order_518692.PDF,200.72,197.780450,197.780450,1,-2.939550,-1.46
96,window_97,15,PO#_STK101225_Order_518692.PDF,200.72,197.780450,197.780450,1,-2.939550,-1.46


In [17]:
df["diff"].abs().mean()


np.float64(14.024782761224492)